# 02_02 - Folds and Forecast Targets

Two building blocks needed before any model gets trained: the 3 expanding-window folds, and the origin-to-horizon pivot that turns the raw hourly consumption series into the actual 24h-ahead targets - in both the long form the classifier needs and the wide form the multi-output regressor needs.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parents[1] / "src"))

import pandas as pd
from common.folds import FOLDS, split_fold, iter_folds
from common.targets import build_long_targets, build_wide_targets

DATA_PATH = Path.cwd().parents[1] / "data" / "interim" / "fsa_hourly_master.parquet"
df = pd.read_parquet(DATA_PATH)

print("shape:", df.shape)

shape: (262944, 47)


## 1. Expanding-window folds

The 3 folds are fixed by the team's methodology: each one trains on everything from 2021 up to a point, then tests on the following year. The training window only ever grows - it never drops the earliest years the way a rolling window would.

In [2]:
pd.DataFrame(FOLDS).T

,train_years,test_year
fold_1,"(2021, 2022)",2023
fold_2,"(2021, 2023)",2024
fold_3,"(2021, 2024)",2025


In [3]:
rows = []
for fold_name, train_df, test_df in iter_folds(df):
    rows.append({
        "fold": fold_name,
        "train_years": sorted(train_df["year"].unique()),
        "train_rows": len(train_df),
        "test_years": sorted(test_df["year"].unique()),
        "test_rows": len(test_df),
    })
pd.DataFrame(rows)

,fold,train_years,train_rows,test_years,test_rows
0,fold_1,"[2021, 2022]",105120,[2023],52560
1,fold_2,"[2021, 2022, 2023]",157680,[2024],52704
2,fold_3,"[2021, 2022, 2023, 2024]",210384,[2025],52560


Each fold's `train_years` and `test_years` line up exactly with what `FOLDS` specifies, with no overlap between train and test in any fold. Row counts grow as expected across folds (fold_2's train is fold_1's train plus one more year, and so on), and fold_2's test year is slightly bigger than the others (52,704 vs 52,560 rows) because 2024 is a leap year - 6 FSAs x 8,784 hours instead of 8,760.

## 2. Origin-to-horizon pivot

The model needs to learn, from any given hour (the "origin"), what consumption actually turns out to be in each of the next 24 hours. Since every FSA is a complete, gap-free hourly grid, "h hours after a given origin" is exactly "h rows later" in that FSA's own series - no timestamp math or joins needed, just shifting the series.

Two shapes come out of the same underlying shift: a long table (one row per origin/horizon pair - what the row-wise classifier needs) and a wide table (one row per origin, with all 24 horizons as separate columns - what the multi-output regressor needs, since it predicts all 24 values from a single row of input at once).

In [4]:
long_df = build_long_targets(df)
wide_df = build_wide_targets(df)

print("long shape:", long_df.shape)
print("wide shape:", wide_df.shape)

long shape: (6307200, 5)
wide shape: (262800, 26)


Checking one concrete origin against the raw hourly values from Day 1 (M5S, 2021-01-01): the first few horizons should just be the next few hours' actual consumption, in order.

In [5]:
example = long_df[(long_df["FSA"] == "M5S") & (long_df["origin_timestamp"] == "2021-01-01 00:00:00")]
example.sort_values("horizon").head(5)

,FSA,origin_timestamp,horizon,forecast_timestamp,actual
87600,M5S,2021-01-01,1,2021-01-01 01:00:00,3868.8
350400,M5S,2021-01-01,2,2021-01-01 02:00:00,3652.6
613200,M5S,2021-01-01,3,2021-01-01 03:00:00,3538.7
876000,M5S,2021-01-01,4,2021-01-01 04:00:00,3417.8
1138800,M5S,2021-01-01,5,2021-01-01 05:00:00,3368.0


In [6]:
wide_df[(wide_df["FSA"] == "M5S") & (wide_df["origin_timestamp"] == "2021-01-01 00:00:00")].iloc[:, :6]

,FSA,origin_timestamp,actual_h1,actual_h2,actual_h3,actual_h4
87600,M5S,2021-01-01,3868.8,3652.6,3538.7,3417.8


Same numbers, two shapes: the long table's horizon 1/2/3 rows for this origin match `actual_h1`/`actual_h2`/`actual_h3` in the wide table exactly (3868.8, 3652.6, 3538.7).

Near the end of each FSA's history there isn't a full 24 hours of future data left, so those origins get dropped - both tables are trimmed the same way, so they end up covering exactly the same set of origins.

In [7]:
long_origins = set(zip(long_df["FSA"], long_df["origin_timestamp"]))
wide_origins = set(zip(wide_df["FSA"], wide_df["origin_timestamp"]))
rows_per_origin = long_df.groupby(["FSA", "origin_timestamp"]).size().unique()

print("long and wide cover the exact same origins:", long_origins == wide_origins)
print("every origin has exactly 24 rows in the long table:", list(rows_per_origin))
print("unique origins:", len(wide_origins))

long and wide cover the exact same origins: True
every origin has exactly 24 rows in the long table: [np.int64(24)]
unique origins: 262800
